# Thử nghiệm Đối chứng với Thư viện Scikit-Surprise

Notebook này sử dụng thư viện `scikit-surprise` để chạy các thuật toán tương tự trên cùng một tập dữ liệu MovieLens 100k.
Kết quả MAE, RMSE, Precision, Recall, F1 và Thời gian dự đoán ở đây sẽ được dùng để đối chiếu trực tiếp với kết quả chạy từ thư mục `src/` (tự code từ đầu).

Các thuật toán cần kiểm chứng:
1. User-Based CF (Basic - Cosine)
2. User-Based CF (Means - Pearson)
3. Item-Based CF (Basic - Cosine)
4. Item-Based CF (Weighted/Adjusted Cosine - trong surprise sử dụng Pearson)
5. SVD (Matrix Factorization)

In [1]:
import os
import time
import numpy as np
import pandas as pd
from surprise import Dataset, Reader, KNNBasic, KNNWithMeans, SVD, accuracy
from surprise.model_selection import train_test_split
import sys
sys.path.append("../")
from src.evaluation import compute_mae, compute_rmse, compute_precision_recall_at_k, compute_f1_at_k, compute_prediction_time
from src.data_loader import load_matrix

# Đọc dữ liệu
data_path = '../data/raw/u.data'
reader = Reader(line_format='user item rating timestamp', sep='\t')
data = Dataset.load_from_file(data_path, reader=reader)

# Chia tập Train/Test (80/20)
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# Load custom train and test matrices for fair comparison
train_matrix = load_matrix("../data/processed/train_matrix.npy")
test_matrix = load_matrix("../data/processed/test_matrix.npy")

class SurpriseWrapper:
    """Wrapper để biến mô hình Surprise có hàm predict_batch giống mô hình custom"""
    def __init__(self, algo):
        self.algo = algo
        
    def predict_batch(self, user_idx, item_indices):
        uid = str(user_idx + 1)
        preds = []
        for iid_idx in item_indices:
            iid = str(iid_idx + 1)
            preds.append(self.algo.predict(uid, iid).est)
        return np.array(preds)

In [2]:
print("=== ĐÁNH GIÁ THƯ VIỆN SURPRISE ===")
surprise_results = {}

algorithms = {
    'User-Based Basic (Cosine)': KNNBasic(k=40, sim_options={'name': 'cosine', 'user_based': True}, verbose=False),
    'User-Based Means (Pearson)': KNNWithMeans(k=40, sim_options={'name': 'pearson', 'user_based': True}, verbose=False),
    'Item-Based Basic (Cosine)': KNNBasic(k=40, sim_options={'name': 'cosine', 'user_based': False}, verbose=False),
    'Item-Based (Adj Cosine / Pearson)': KNNBasic(k=40, sim_options={'name': 'pearson', 'user_based': False}, verbose=False),
    'SVD': SVD(n_factors=20, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
}

for name, algo in algorithms.items():
    print(f"\nĐang chạy: {name}...")
    start_train = time.time()
    algo.fit(trainset)
    
    preds = algo.test(testset)
    rmse = accuracy.rmse(preds, verbose=False)
    mae = accuracy.mae(preds, verbose=False)
    
    wrapper = SurpriseWrapper(algo)
    precision, recall = compute_precision_recall_at_k(train_matrix, test_matrix, wrapper, k=10)
    f1 = compute_f1_at_k(precision, recall)
    pred_time = compute_prediction_time(test_matrix, wrapper)
    
    surprise_results[name] = {
        'RMSE': rmse,
        'MAE': mae,
        'Precision@10': precision,
        'Recall@10': recall,
        'F1@10': f1,
        'Time/User (s)': pred_time
    }
    
    print(f"RMSE: {rmse:.4f} | MAE: {mae:.4f} | P@10: {precision:.4f} | R@10: {recall:.4f} | Time: {pred_time:.4f}s")

=== ĐÁNH GIÁ THƯ VIỆN SURPRISE ===

Đang chạy: User-Based Basic (Cosine)...
RMSE: 1.0194 | MAE: 0.8038 | P@10: 0.0020 | R@10: 0.0020 | Time: 0.0015s

Đang chạy: User-Based Means (Pearson)...
RMSE: 0.9492 | MAE: 0.7430 | P@10: 0.0071 | R@10: 0.0061 | Time: 0.0018s

Đang chạy: Item-Based Basic (Cosine)...
RMSE: 1.0264 | MAE: 0.8104 | P@10: 0.0859 | R@10: 0.0433 | Time: 0.0013s

Đang chạy: Item-Based (Adj Cosine / Pearson)...
RMSE: 1.0411 | MAE: 0.8332 | P@10: 0.0040 | R@10: 0.0022 | Time: 0.0012s

Đang chạy: SVD...
RMSE: 0.9350 | MAE: 0.7370 | P@10: 0.0909 | R@10: 0.0776 | Time: 0.0001s


In [3]:
print("\n======================================================")
print("=== ĐỐI CHỨNG VỚI MÔ HÌNH TỰ XÂY DỰNG TỪ ĐẦU (SCRATCH) ===")
print("======================================================\n")
import pickle
from src.recommender import MatrixFactorizationSVD, UserBasedCollaborativeFiltering, ItemBasedCollaborativeFiltering

with open("../models/user_cf.pkl", "rb") as f:
    user_cf = pickle.load(f)
with open("../models/item_cf.pkl", "rb") as f:
    item_cf = pickle.load(f)
    
svd_model = MatrixFactorizationSVD()
svd_model.load_model("../models/svd_weights.pkl")
svd_model.train_matrix = train_matrix

custom_algos = {
    'User-Based Basic (Cosine)': UserBasedCollaborativeFiltering(k_neighbors=user_cf.k_neighbors, prediction_mode='basic'),
    'User-Based Means (Pearson)': UserBasedCollaborativeFiltering(k_neighbors=user_cf.k_neighbors, prediction_mode='means'),
    'Item-Based Basic (Cosine)': ItemBasedCollaborativeFiltering(k_neighbors=item_cf.k_neighbors),
    'Item-Based (Adj Cosine / Pearson)': ItemBasedCollaborativeFiltering(k_neighbors=item_cf.k_neighbors),
    'SVD': svd_model
}

# Attach dependencies for CF models
custom_algos['User-Based Basic (Cosine)'].train_matrix = user_cf.train_matrix
custom_algos['User-Based Basic (Cosine)'].similarity_matrix = user_cf.cosine_similarity_matrix
custom_algos['User-Based Basic (Cosine)'].user_means = user_cf.user_means

custom_algos['User-Based Means (Pearson)'].train_matrix = user_cf.train_matrix
custom_algos['User-Based Means (Pearson)'].similarity_matrix = user_cf.pearson_similarity_matrix
custom_algos['User-Based Means (Pearson)'].user_means = user_cf.user_means

custom_algos['Item-Based Basic (Cosine)'].train_matrix = item_cf.train_matrix
custom_algos['Item-Based Basic (Cosine)'].similarity_matrix = item_cf.cosine_similarity_matrix

custom_algos['Item-Based (Adj Cosine / Pearson)'].train_matrix = item_cf.train_matrix
custom_algos['Item-Based (Adj Cosine / Pearson)'].similarity_matrix = item_cf.adjusted_cosine_similarity_matrix

custom_results = {}

for name, model in custom_algos.items():
    print(f"\nĐang đánh giá Custom: {name}...")
    rmse = compute_rmse(test_matrix, model)
    mae = compute_mae(test_matrix, model)
    precision, recall = compute_precision_recall_at_k(train_matrix, test_matrix, model, k=10)
    f1 = compute_f1_at_k(precision, recall)
    pred_time = compute_prediction_time(test_matrix, model)
    
    custom_results[name] = {
        'RMSE': rmse,
        'MAE': mae,
        'Precision@10': precision,
        'Recall@10': recall,
        'F1@10': f1,
        'Time/User (s)': pred_time
    }
    
    print(f"RMSE: {rmse:.4f} | MAE: {mae:.4f} | P@10: {precision:.4f} | R@10: {recall:.4f} | Time: {pred_time:.4f}s")


=== ĐỐI CHỨNG VỚI MÔ HÌNH TỰ XÂY DỰNG TỪ ĐẦU (SCRATCH) ===


Đang đánh giá Custom: User-Based Basic (Cosine)...
RMSE: 1.0101 | MAE: 0.8048 | P@10: 0.0000 | R@10: 0.0000 | Time: 0.0005s

Đang đánh giá Custom: User-Based Means (Pearson)...
RMSE: 0.9376 | MAE: 0.7324 | P@10: 0.0061 | R@10: 0.0050 | Time: 0.0006s

Đang đánh giá Custom: Item-Based Basic (Cosine)...
RMSE: 0.9855 | MAE: 0.7752 | P@10: 0.0323 | R@10: 0.0122 | Time: 0.0004s

Đang đánh giá Custom: Item-Based (Adj Cosine / Pearson)...
RMSE: 0.9450 | MAE: 0.7411 | P@10: 0.0242 | R@10: 0.0113 | Time: 0.0004s

Đang đánh giá Custom: SVD...
RMSE: 0.9388 | MAE: 0.7421 | P@10: 0.0758 | R@10: 0.0612 | Time: 0.0000s


### Bảng So Sánh Hiệu Năng Chi Tiết (Surprise vs Custom)
Tổng hợp lại toàn bộ kết quả vào một bảng để so sánh chi tiết hiệu suất giữa thư viện và thuật toán tự code từ đầu.

In [4]:
df_data = []
for name in algorithms.keys():
    s_res = surprise_results[name]
    c_res = custom_results[name]
    df_data.append({
        'Mô hình': name,
        'Surprise RMSE': s_res['RMSE'], 'Custom RMSE': c_res['RMSE'],
        'Surprise MAE': s_res['MAE'], 'Custom MAE': c_res['MAE'],
        'Surprise Prec@10': s_res['Precision@10'], 'Custom Prec@10': c_res['Precision@10'],
        'Surprise Rec@10': s_res['Recall@10'], 'Custom Rec@10': c_res['Recall@10'],
        'Surprise F1@10': s_res['F1@10'], 'Custom F1@10': c_res['F1@10'],
        'Surprise Time(s)': s_res['Time/User (s)'], 'Custom Time(s)': c_res['Time/User (s)']
    })

comparison_df = pd.DataFrame(df_data)
from IPython.display import display
display(comparison_df.round(4))

,Mô hình,Surprise RMSE,Custom RMSE,Surprise MAE,Custom MAE,Surprise Prec@10,Custom Prec@10,Surprise Rec@10,Custom Rec@10,Surprise F1@10,Custom F1@10,Surprise Time(s),Custom Time(s)
0,User-Based Basic (Cosine),1.0194,1.0101,0.8038,0.8048,0.0020,0.0000,0.0020,0.0000,0.0020,0.0000,0.0015,0.0005
1,User-Based Means (Pearson),0.9492,0.9376,0.7430,0.7324,0.0071,0.0061,0.0061,0.0050,0.0066,0.0055,0.0018,0.0006
2,Item-Based Basic (Cosine),1.0264,0.9855,0.8104,0.7752,0.0859,0.0323,0.0433,0.0122,0.0576,0.0177,0.0013,0.0004
3,Item-Based (Adj Cosine / Pearson),1.0411,0.9450,0.8332,0.7411,0.0040,0.0242,0.0022,0.0113,0.0028,0.0154,0.0012,0.0004
4,SVD,0.9350,0.9388,0.7370,0.7421,0.0909,0.0758,0.0776,0.0612,0.0837,0.0677,0.0001,0.0000


### So sánh thực tế: Top 5 Gợi ý phim cho User 1 (dùng mô hình SVD)
Để thấy rõ hơn, chúng ta thử xem danh sách top 5 phim mà mô hình tự làm và thư viện đề xuất cho người dùng có id là 1 có tương đồng không.

In [5]:
from src.data_loader import load_movie_titles

movie_titles = load_movie_titles("../data/raw/u.item")
user_id = 1
user_idx = user_id - 1

print("=== [Custom SVD] Top 5 Phim Gợi Ý ===")
unviewed_items = np.where(train_matrix[user_idx] == 0)[0]
custom_preds = svd_model.predict_batch(user_idx, unviewed_items)
top_5_idx_custom = unviewed_items[np.argsort(custom_preds)[-5:][::-1]]

for rank, idx in enumerate(top_5_idx_custom, 1):
    m_id = int(idx) + 1
    name = movie_titles.get(m_id, f"Phim {m_id}")
    pred_score = custom_preds[np.where(unviewed_items == idx)[0][0]]
    print(f"{rank}. {name} (Dự đoán: {pred_score:.2f} sao)")

print("\n=== [Surprise SVD] Top 5 Phim Gợi Ý ===")
try:
    user_inner_id = trainset.to_inner_uid(str(user_id))
    viewed_inner_items = set([j for (j, r) in trainset.ur[user_inner_id]])
    
    surprise_preds = []
    for i in trainset.all_items():
        if i not in viewed_inner_items:
            item_raw_id = trainset.to_raw_iid(i)
            pred = algorithms['SVD'].predict(uid=str(user_id), iid=item_raw_id).est
            surprise_preds.append((int(item_raw_id), pred))
            
    surprise_preds.sort(key=lambda x: x[1], reverse=True)
    top_5_surprise = surprise_preds[:5]
    
    for rank, (m_id, pred_score) in enumerate(top_5_surprise, 1):
        name = movie_titles.get(int(m_id), f"Phim {m_id}")
        print(f"{rank}. {name} (Dự đoán: {pred_score:.2f} sao)")
except ValueError:
    print(f"Không thể tìm thấy user_id {user_id} trong trainset của Surprise.")

=== [Custom SVD] Top 5 Phim Gợi Ý ===
1. Schindler's List (1993) (Dự đoán: 4.81 sao)
2. Boot, Das (1981) (Dự đoán: 4.58 sao)
3. Casablanca (1942) (Dự đoán: 4.51 sao)
4. Sunset Blvd. (1950) (Dự đoán: 4.49 sao)
5. Rear Window (1954) (Dự đoán: 4.48 sao)

=== [Surprise SVD] Top 5 Phim Gợi Ý ===
1. Close Shave, A (1995) (Dự đoán: 4.73 sao)
2. Schindler's List (1993) (Dự đoán: 4.65 sao)
3. L.A. Confidential (1997) (Dự đoán: 4.60 sao)
4. It's a Wonderful Life (1946) (Dự đoán: 4.59 sao)
5. One Flew Over the Cuckoo's Nest (1975) (Dự đoán: 4.59 sao)
